# SELFIE on K2-V2 writer/editor agents (LangGraph)

Minimum-viable run. Short prompts, greedy decoding, a few probe layers.

Before running, from a terminal in this project directory:

```bash
pip install 'transformers>=4.45,<4.60' accelerate bitsandbytes langgraph pandas
```

**Hardware**: K2-V2 is 70B. At bf16 you need ~140GB of VRAM across GPUs
(e.g. 2×A100-80GB or 4×A6000-48GB). The notebook uses `device_map="auto"` so
HF will shard across whatever is visible. If you only have one smaller GPU,
set `dtype=torch.float16` and add `load_in_4bit=True` via BitsAndBytesConfig
(left as an exercise — see the commented block below).

In [ ]:
import sys, os
# Make sure we can import the local modules.
sys.path.insert(0, os.path.abspath('.'))

import torch
import pandas as pd
pd.set_option('display.max_colwidth', 120)

from k2v2_backend import K2V2Backend
from agents_graph import make_graph

In [ ]:
# Load K2-V2-Instruct once. Takes several minutes the first time (download + load).
backend = K2V2Backend(
    model_name="LLM360/K2-V2-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
)

# --- if you need 4-bit quantization to fit on smaller GPUs, replace the above with: ---
# from transformers import BitsAndBytesConfig
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
#                          bnb_4bit_quant_type="nf4")
# # then monkey-patch K2V2Backend.__init__ or load manually and assign backend.model

## Sanity check: plain generation works

In [ ]:
ids = backend.build_chat_prompt(
    system="You are K2, a helpful assistant.",
    user="Say hi in 5 words.",
    reasoning_effort="medium",
)
r = backend.generate(ids, max_new_tokens=20, capture_hidden=False)
print(r.output_text)

## Run the writer → editor graph

Short task on purpose. Writer is capped at 40 tokens, editor at 60.

In [ ]:
graph = make_graph(backend, max_writer_tokens=40, max_editor_tokens=60)

final = graph.invoke({"task": "Describe a golden retriever in 3 sentences."})

In [ ]:
print("=== WRITER OUTPUT ===")
print(final['writer_output_text'])
print()
print("=== EDITOR VERDICT (normal, text-passthrough) ===")
print(final['editor_verdict'])
print()
print("=== EDITOR VERDICT (writer hidden states injected at placeholders) ===")
print(final['editor_injected_verdict'])

## Experiment 1 — SELFIE on the writer's output tokens

Each row is one `(layer, token_idx)` probe on the writer's own hidden
states at the position where it was generating token `token_idx`. The
`token` column is what the writer actually emitted there; the
`interpretation` column is the model's gloss of what was encoded in the
hidden state at the chosen layer.

In [ ]:
final['selfie_writer']

## Experiment 3 — SELFIE on the editor's hidden states over the draft span

Same structure, but now we're reading out what the EDITOR's hidden states
encode at the positions where it was reading the writer's tokens.

In [ ]:
final['selfie_editor_on_draft']

## Side-by-side: writer's self-interpretation vs editor's interpretation of the same token

Join the two dataframes on `(layer, relative_position)` where relative
position is the offset within the writer's output span.

In [ ]:
# Compute relative positions.
wr = final['writer_result']
writer_first = wr.prompt_len
draft_first = final['editor_draft_start']

w_df = final['selfie_writer'].copy()
w_df['rel_pos'] = w_df['token_idx'] - writer_first
w_df = w_df.rename(columns={'token': 'writer_token', 'interpretation': 'writer_interp'})

e_df = final['selfie_editor_on_draft'].copy()
e_df['rel_pos'] = e_df['token_idx'] - draft_first
e_df = e_df.rename(columns={'token': 'editor_token', 'interpretation': 'editor_interp'})

merged = w_df.merge(
    e_df[['layer', 'rel_pos', 'editor_token', 'editor_interp']],
    on=['layer', 'rel_pos'],
    how='inner',
)
merged = merged[['layer', 'rel_pos', 'writer_token', 'editor_token',
                 'writer_interp', 'editor_interp']]
merged

## What to look at

- **Same `(layer, rel_pos)` rows**: does `writer_interp` roughly agree with
  `editor_interp`? Divergence at a position = something about how the editor
  reinterprets that token.
- **Layer sweep**: lower layers → surface/lexical, higher → semantic. The
  late-layer interpretations should read more abstractly.
- **Experiment 2 (injected) vs normal editor verdict**: if they're close,
  the writer's final hidden states are essentially equivalent to its text.
  If they diverge, the hidden state carries something the surface tokens
  don't — interesting signal.

If Experiment 2's output is incoherent, try `inject_layer=1` or a middle
layer instead of `0` (currently hard-coded in `agents_graph.injected_editor_node`
— easiest place to experiment). Final-layer-of-writer → layer-0-of-editor is
a hard jump across the residual stream.